# Data Preparation Pipeline — กระบวนการสกัดและจัดเตรียมข้อมูลรัฐธรรมนูญ

- **รายวิชา CPE232 Data Models**
- **กลุ่ม Final Project: ทีมงานชุมชนนิมนต์ยิ้ม ss67**
- **หัวข้อ Final Project: (Twenty Constitutions Analysis) Analyze of 20 Thai Constitutions**

---

## ส่วนนำ (Overview)

เอกสารฉบับนี้อธิบายถึงขั้นตอนและกระบวนการในการดึงข้อมูล (Data Preparation) สำหรับเอกสารรัฐธรรมนูญแห่งราชอาณาจักรไทยจำนวน 38 ฉบับ ตั้งแต่ปี พ.ศ. 2475 ถึง 2564 เพื่อนำไปใช้เป็นชุดข้อมูลสำหรับกระบวนการ Data Modeling 

เอกสารต้นฉบับแบ่งออกเป็น 2 ประเภทหลัก ๆ ตามลักษณะของไฟล์ PDF ดังนี้:

- **เอกสารภาพสแกน (ไฟล์ลำดับที่ 1–32)**: ลักษณะเป็นไฟล์ภาพที่ผ่านการสแกน ไม่สามารถคัดลอกข้อความได้โดยตรง จึงต้องใช้กระบวนการ Optical Character Recognition (OCR) ผ่าน Typhoon OCR API
- **เอกสารข้อความ (ไฟล์ลำดับที่ 33–38)**: ลักษณะเป็นไฟล์ PDF ที่สามารถคัดลอกข้อความได้โดยตรง โดยจะใช้ไลบรารี PyMuPDF และ pdfplumber ในการสกัดข้อความ

### รายการตรวจสอบก่อนการรัน (Pre-flight Checklist)

ก่อนดำเนินการรันเซลล์ในเอกสารนี้ กรุณาตรวจสอบการตั้งค่าดังนี้:

| ลำดับ | รายการ | วิธีการตั้งค่า |
|---|------|-----------------|
| 1 | **การเชื่อมต่ออินเทอร์เน็ต** | ไปที่ Notebook settings → เลือก Internet → เปิดใช้งาน (Toggle ON) |
| 2 | **การเพิ่มชุดข้อมูล PDF** | ไปที่ Settings → Data → เลือก add dataset และระบุ `thai-constitutions-pdfs` (ต้องมีครบ 38 ไฟล์) |
| 3 | **การตั้งค่า API Key** | ไปที่ Settings → Secrets → เพิ่มกุญแจเข้ารหัสชื่อ `TYPHOON_OCR_API_KEY` และเปิดใช้งาน |
| 4 | **ตัวประมวลผล (Accelerator)** | ตั้งค่าเป็น None (CPU) เนื่องจากกระบวนการไม่อาศัย GPU |

### โครงสร้างไฟล์ผลลัพธ์ที่คาดหวัง

```
/kaggle/working/
├── ocr_cache/      ← โฟลเดอร์เก็บข้อมูล OCR ชั่วคราวรายหน้า (รองรับการรันต่อหากระบบหยุดทำงาน)
├── extracted_text/ ← โฟลเดอร์เก็บข้อมูลข้อความเบื้องต้นสำหรับเอกสารประเภทข้อความ
├── processed/      ← โฟลเดอร์เก็บไฟล์ JSON ฉบับสมบูรณ์ (1 ไฟล์ต่อ 1 ฉบับ)
└── logs/           ← โฟลเดอร์เก็บข้อมูลบันทึกข้อผิดพลาด
```

> หมายเหตุ: กระบวนการ Typhoon OCR กำหนดขีดจำกัดไว้ที่ประมาณ 20 หน้าต่อนาที ดังนั้นการประมวลผลเอกสารภาพสแกนทั้งหมด 32 ฉบับ อาจใช้เวลาประมาณ 8 - 14 ชั่วโมง ขึ้นอยู่กับจำนวนหน้าโดยรวม

---
## ส่วนที่ 1: การตั้งค่าสภาพแวดล้อมระบบ (Environment Setup)

### 1.1 การตรวจสอบสภาพแวดล้อมและการเชื่อมต่ออินเทอร์เน็ต

ในส่วนนี้จะเป็นการตรวจสอบว่าระบบกำลังทำงานอยู่บนสภาพแวดล้อมใด (Kaggle, Google Colab หรือ เครื่องคอมพิวเตอร์ส่วนตัว) เนื่องจากแต่ละพื้นที่มีการจัดการเส้นทางไฟล์และระบบรักษาความปลอดภัยของรหัสผ่านที่แตกต่างกัน พร้อมทั้งตรวจสอบการเชื่อมต่ออินเทอร์เน็ตเบื้องต้น เนื่องจาก Typhoon OCR API จำเป็นต้องมีการส่งรับข้อมูลเครือข่าย

In [1]:
import os, sys, socket
from pathlib import Path

# Detect the current runtime environment.
IS_KAGGLE = os.path.exists('/kaggle/working')
IS_COLAB  = False
try:
    import google.colab
    IS_COLAB = True
except ImportError:
    pass
IS_LOCAL = not IS_KAGGLE and not IS_COLAB
ENV = 'Kaggle' if IS_KAGGLE else ('Colab' if IS_COLAB else 'Local')

print(f"Environment : {ENV}")
print(f"Python      : {sys.version.split()[0]}")

# Check internet connectivity by attempting a TCP connection to a known host.
def _check_internet() -> bool:
    try:
        socket.setdefaulttimeout(5)
        socket.socket(socket.AF_INET, socket.SOCK_STREAM).connect(("8.8.8.8", 53))
        return True
    except OSError:
        return False

INTERNET_OK = _check_internet()
print(f"\nInternet    : {'OK' if INTERNET_OK else 'No connection — check Kaggle settings'}")

if not INTERNET_OK and IS_KAGGLE:
    print()
    print("=" * 60)
    print("Internet access is required to run the OCR pipeline.")
    print("Go to: Notebook settings -> Internet -> enable the toggle")
    print("Then save and restart the notebook from the beginning.")
    print("=" * 60)
    raise SystemExit("Please enable Internet Access before running.")

Environment : Kaggle
Python      : 3.12.12

Internet    : OK


### 1.2 การติดตั้งไลบรารีและแพ็กเกจที่จำเป็น

ในเซลล์นี้จะดำเนินการติดตั้งซอฟต์แวร์ระดับระบบและไลบรารีของ Python โดยจะมีการติดตั้ง `poppler-utils` ซึ่งเป็นองค์ประกอบพื้นฐานสำหรับ `typhoon-ocr` รวมไปถึงแพ็กเกจฟอนต์ภาษาไทยเพื่อใช้ในการวาดกราฟผ่าน matplotlib ได้อย่างถูกต้อง

การกำหนดเวอร์ชัน `typhoon-ocr==0.4.1` มีจุดประสงค์เพื่อควบคุมความเสถียร เนื่องจากระบบอาจมีพฤติกรรมเปลี่ยนแปลงได้หากมีการใช้ไลบรารีเวอร์ชันใหม่กว่าโดยไม่ได้ตรวจสอบ

In [2]:
import subprocess

# Install system-level packages required by typhoon-ocr and matplotlib.
print("[1/3] Installing system packages (poppler-utils, Thai font)...")
r = subprocess.run(
    ["apt-get", "install", "-y", "-q",
     "poppler-utils",
     "poppler-data",
     "fonts-thai-tlwg"],
    capture_output=True, text=True
)
print("  OK" if r.returncode == 0 else f"  Warning: {r.stderr[:300]}")

# Install Python packages.
# typhoon-ocr 0.4.1 is the latest available version on PyPI.
print("\n[2/3] Installing Python packages...")
PACKAGES = [
    "typhoon-ocr==0.4.1",
    "pymupdf>=1.24.0",
    "pdfplumber>=0.11.0",
    "tqdm>=4.66.0",
    "tenacity>=8.2.0",
    "pandas>=2.0.0",
    "python-dotenv>=1.0.0",
]
r = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q"] + PACKAGES,
    capture_output=True, text=True
)
if r.returncode != 0:
    print(f"  pip install failed:\n{r.stderr[-800:]}")
    raise RuntimeError("pip install failed")
print("  OK")

# Configure matplotlib to use a Thai-capable font so that Thai text
# renders correctly in any charts or visualizations we produce later.
print("\n[3/3] Configuring matplotlib for Thai text rendering...")
import matplotlib
matplotlib.font_manager._load_fontmanager(try_read_cache=False)
from matplotlib import font_manager
import matplotlib.pyplot as plt

thai_font_found = False
for fw in font_manager.findSystemFonts(fontext='ttf'):
    if any(x in fw.lower() for x in ['thai', 'tlwg', 'norasi', 'loma', 'garuda', 'kinnari']):
        font_manager.fontManager.addfont(fw)
        thai_font_found = True
        print(f"  Thai font loaded: {Path(fw).name}")
        break

if not thai_font_found:
    print("  Warning: no Thai font found — Thai text in charts may appear as boxes")

plt.rcParams.update({
    'font.family':        ['Garuda', 'Norasi', 'Loma', 'TH Sarabun New', 'DejaVu Sans'],
    'axes.unicode_minus': False,
    'figure.dpi':         100,
})
print("  matplotlib configured")

[1/3] Installing system packages (poppler-utils, Thai font)...
  OK

[2/3] Installing Python packages...
  OK

[3/3] Configuring matplotlib for Thai text rendering...
  Thai font loaded: Kinnari-BoldOblique.ttf
  matplotlib configured


### 1.3 การจัดการเส้นทางและไดเรกทอรีของไฟล์

รหัสส่วนนี้จะทำหน้าที่กำหนดเส้นทางสำหรับการอ่านและบันทึกไฟล์ โดยในระบบของ Kaggle ข้อมูลชุดข้อความนำเข้าจะอยู่ในสถานะอ่านได้อย่างเดียว (Read-only) ภายใต้ `/kaggle/input/` และไฟล์ผลลัพธ์จะต้องบันทึกไปที่ `/kaggle/working/` ระบบจะทำการค้นหาที่อยู่ของไฟล์เอกสารรัฐธรรมนูญโดยอัตโนมัติ เพื่อป้องกันปัญหาที่เกิดจากการระบุเส้นทางแบบตายตัว

In [3]:
if IS_KAGGLE:
    WORK_DIR = Path("/kaggle/working")

    # Auto-discover the dataset directory that contains PDF files.
    # This avoids hardcoding the exact Kaggle dataset slug.
    input_root = Path("/kaggle/input/datasets/wisitsuwannao")
    pdf_candidates = []
    for d in input_root.rglob("*.pdf"):
        parent = d.parent
        if parent not in [c[0] for c in pdf_candidates]:
            count = len(list(parent.glob("*.pdf")))
            if count > 0:
                pdf_candidates.append((parent, count))

    if pdf_candidates:
        pdf_candidates.sort(key=lambda x: x[1], reverse=True)
        PDF_DIR = pdf_candidates[0][0]
        print(f"PDF dataset found: {PDF_DIR} ({pdf_candidates[0][1]} files)")
    else:
        PDF_DIR = input_root / "thai-constitutions-pdfs"
        print("Warning: no PDF files found under /kaggle/input/")
        print("Add the 'thai-constitutions-pdfs' dataset via Settings -> Data")

elif IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = Path("/content/drive/MyDrive/CPE232_Constitution")
    PDF_DIR     = PROJECT_DIR / "raw_pdfs"
    WORK_DIR    = PROJECT_DIR

else:
    BASE     = Path(".").resolve()
    PDF_DIR  = BASE / "data" / "raw_pdfs"
    WORK_DIR = BASE

# Create output directories.
OCR_CACHE_DIR = WORK_DIR / "ocr_cache"
TEXT_DIR      = WORK_DIR / "extracted_text"
PROCESSED_DIR = WORK_DIR / "processed"
LOG_DIR       = WORK_DIR / "logs"

for d in [OCR_CACHE_DIR, TEXT_DIR, PROCESSED_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"\nPDF_DIR      : {PDF_DIR}")
print(f"OCR_CACHE    : {OCR_CACHE_DIR}")
print(f"PROCESSED    : {PROCESSED_DIR}")

if PDF_DIR.exists():
    all_pdfs = sorted(PDF_DIR.glob("*.pdf"))
    print(f"\nPDF files found: {len(all_pdfs)}/38")
    if len(all_pdfs) < 38:
        print(f"  Warning: {38 - len(all_pdfs)} files missing — check dataset")
else:
    print(f"\nWarning: {PDF_DIR} does not exist")
    all_pdfs = []

PDF dataset found: /kaggle/input/datasets/wisitsuwannao/thai-constitutions-pdfs (38 files)

PDF_DIR      : /kaggle/input/datasets/wisitsuwannao/thai-constitutions-pdfs
OCR_CACHE    : /kaggle/working/ocr_cache
PROCESSED    : /kaggle/working/processed

PDF files found: 38/38


### 1.4 การเตรียมใช้งาน API Key

กระบวนการใช้ Typhoon OCR API จำเป็นต้องมีการยืนยันตัวตนผ่าน API Key สำหรับระบบ Kaggle จะทำการดึงข้อมูลผ่านเครื่องมือ `UserSecretsClient` ส่วนใน Colab จะดึงผ่าน `userdata` และหากรันในคอมพิวเตอร์ส่วนตัวจะอาศัยไฟล์ `.env`

เมื่อดึงรหัสได้สำเร็จ ระบบจะกำหนดค่าให้ตัวแปรสภาพแวดล้อม `TYPHOON_OCR_API_KEY` เพื่อให้ตัวไลบรารี `typhoon-ocr` สามารถทำงานร่วมกันได้อัตโนมัติ

In [4]:
api_key = None

if IS_KAGGLE:
    try:
        from kaggle_secrets import UserSecretsClient
        api_key = UserSecretsClient().get_secret("TYPHOON_OCR_API_KEY")
        print("API key loaded from Kaggle Secrets")
    except Exception as e:
        print(f"Could not load API key from Kaggle Secrets: {e}")
        print("  Go to: Settings -> Secrets -> add TYPHOON_OCR_API_KEY")
        print("  Make sure the toggle is enabled for this notebook")

elif IS_COLAB:
    try:
        from google.colab import userdata
        api_key = userdata.get("TYPHOON_OCR_API_KEY")
        print("API key loaded from Colab Secrets")
    except Exception as e:
        print(f"Could not load API key from Colab: {e}")

else:
    from dotenv import load_dotenv
    load_dotenv()
    api_key = os.getenv("TYPHOON_OCR_API_KEY")
    if api_key:
        print("API key loaded from .env file")
    else:
        print("Warning: TYPHOON_OCR_API_KEY not found in .env")

if api_key:
    os.environ["TYPHOON_OCR_API_KEY"] = api_key
    # Show only a partial key for verification without exposing the full secret.
    masked = api_key[:6] + "*" * max(0, len(api_key) - 10) + api_key[-4:]
    print(f"  Key (masked): {masked}")
else:
    print("\nWarning: no API key available.")
    print("  OCR (Step 2) will fail without a valid key.")
    print("  Text extraction (Step 3) does not require a key and will still work.")

API key loaded from Kaggle Secrets
  Key (masked): sk-2OZ*****************************************5K5b


### 1.5 ข้อมูล Metadata และการตรวจสอบความพร้อม

ในเซลล์นี้ ระบบจะทำการกำหนดตัวแปรโครงสร้างข้อมูลสำหรับเอกสารแต่ละฉบับ ตั้งแต่แฟ้มข้อมูลลำดับที่ 1 ถึง 38 ข้อมูลดังกล่าวประกอบไปด้วย ปีพุทธศักราช ปีคริสต์ศักราช ชื่อย่อ ประเภทของแหล่งข้อมูล ยุคสมัยทางประวัติศาสตร์ และประเภทระบอบการปกครอง 

หลังจากการตั้งค่านี้เสร็จสิ้น ระบบจะเช็คความถูกต้องสุดท้ายในสามประเด็น ได้แก่: ความครบถ้วนของไฟล์เอกสารทั้ง 38 ฉบับ, การมีอยู่ของ API key, และความเสถียรของการเชื่อมต่ออินเทอร์เน็ต

In [5]:
# Metadata for all 38 constitutions.
# Format: file_number -> (id, year_th, year_ce, name_short, source_type, era, regime_type)
FILE_MAP = {
     1: ("const_2475",  2475, 1932, "Constitution 2475",            "image_pdf", "early_democracy",   "civilian"),
     2: ("const_2482",  2482, 1939, "Constitution 2482",            "image_pdf", "early_democracy",   "civilian"),
     3: ("const_2483",  2483, 1940, "Constitution 2483",            "image_pdf", "early_democracy",   "civilian"),
     4: ("const_2485",  2485, 1942, "Constitution 2485",            "image_pdf", "early_democracy",   "military"),
     5: ("const_2489",  2489, 1946, "Constitution 2489",            "image_pdf", "early_democracy",   "civilian"),
     6: ("const_2490a", 2490, 1947, "Constitution 2490 (interim)",  "image_pdf", "post_coup_1947",    "military"),
     7: ("const_2490b", 2490, 1947, "Constitution 2490 (amended)",  "image_pdf", "post_coup_1947",    "military"),
     8: ("const_2491a", 2491, 1948, "Constitution 2491 (No.2)",     "image_pdf", "post_coup_1947",    "military"),
     9: ("const_2491b", 2491, 1948, "Constitution 2491 (No.3)",     "image_pdf", "post_coup_1947",    "military"),
    10: ("const_2492",  2492, 1949, "Constitution 2492",            "image_pdf", "post_coup_1947",    "civilian"),
    11: ("const_2495",  2495, 1952, "Constitution 2495",            "image_pdf", "post_coup_1947",    "military"),
    12: ("const_2502",  2502, 1959, "Constitution 2502",            "image_pdf", "dictatorship",      "military"),
    13: ("const_2511",  2511, 1968, "Constitution 2511",            "image_pdf", "dictatorship",      "military"),
    14: ("const_2515",  2515, 1972, "Constitution 2515",            "image_pdf", "dictatorship",      "military"),
    15: ("const_2517",  2517, 1974, "Constitution 2517",            "image_pdf", "democratic_spring", "civilian"),
    16: ("const_2518",  2518, 1975, "Constitution 2518",            "image_pdf", "democratic_spring", "civilian"),
    17: ("const_2519",  2519, 1976, "Constitution 2519",            "image_pdf", "democratic_spring", "military"),
    18: ("const_2520",  2520, 1977, "Constitution 2520",            "image_pdf", "semi_democracy",    "military"),
    19: ("const_2521",  2521, 1978, "Constitution 2521",            "image_pdf", "semi_democracy",    "semi_military"),
    20: ("const_2528",  2528, 1985, "Constitution 2528",            "image_pdf", "semi_democracy",    "semi_military"),
    21: ("const_2532",  2532, 1989, "Constitution 2532",            "image_pdf", "semi_democracy",    "semi_military"),
    22: ("const_2534a", 2534, 1991, "Constitution 2534 (interim)", "image_pdf", "modern_democracy",  "military"),
    23: ("const_2534b", 2534, 1991, "Constitution 2534",           "image_pdf", "modern_democracy",  "military"),
    24: ("const_2535a", 2535, 1992, "Constitution 2535 (No.1)",    "image_pdf", "modern_democracy",  "civilian"),
    25: ("const_2535b", 2535, 1992, "Constitution 2535 (No.2)",    "image_pdf", "modern_democracy",  "civilian"),
    26: ("const_2535c", 2535, 1992, "Constitution 2535 (No.3)",    "image_pdf", "modern_democracy",  "civilian"),
    27: ("const_2535d", 2535, 1992, "Constitution 2535 (No.4)",    "image_pdf", "modern_democracy",  "civilian"),
    28: ("const_2538",  2538, 1995, "Constitution 2538",            "image_pdf", "modern_democracy",  "civilian"),
    29: ("const_2539",  2539, 1996, "Constitution 2539",            "image_pdf", "modern_democracy",  "civilian"),
    30: ("const_2540",  2540, 1997, "Constitution 2540",            "image_pdf", "modern_democracy",  "civilian"),
    31: ("const_2548",  2548, 2005, "Constitution 2548",            "image_pdf", "modern_democracy",  "civilian"),
    32: ("const_2549",  2549, 2006, "Constitution 2549 (interim)",  "image_pdf", "post_coup_2006",    "military"),
    33: ("const_2550",  2550, 2007, "Constitution 2550",            "text_pdf",  "post_coup_2006",    "civilian"),
    34: ("const_2554a", 2554, 2011, "Constitution 2554 (No.1)",    "text_pdf",  "post_coup_2006",    "civilian"),
    35: ("const_2554b", 2554, 2011, "Constitution 2554 (No.2)",    "text_pdf",  "post_coup_2006",    "civilian"),
    36: ("const_2557",  2557, 2014, "Constitution 2557 (interim)",  "text_pdf",  "post_coup_2014",    "military"),
    37: ("const_2560",  2560, 2017, "Constitution 2560",            "text_pdf",  "post_coup_2014",    "semi_military"),
    38: ("const_2564",  2564, 2021, "Constitution 2564",            "text_pdf",  "post_coup_2014",    "semi_military"),
}

# Build the CONSTITUTIONS list from the actual PDF files found in the dataset directory.
CONSTITUTIONS = []
for pdf_path in sorted(PDF_DIR.glob("*.pdf") if PDF_DIR.exists() else []):
    parts = pdf_path.stem.split("_", 1)
    if not parts[0].isdigit():
        continue
    n = int(parts[0])
    if n not in FILE_MAP:
        print(f"  Warning: no metadata for file {n}: {pdf_path.name}")
        continue
    cid, yr_th, yr_ce, name_short, src_type, era, regime = FILE_MAP[n]
    CONSTITUTIONS.append({
        "file_num": n, "id": cid, "year_th": yr_th, "year_ce": yr_ce,
        "name_short": name_short, "source_type": src_type,
        "era": era, "regime_type": regime, "filename": pdf_path.name,
    })

IMAGE_PDFS = [c for c in CONSTITUTIONS if c["source_type"] == "image_pdf"]
TEXT_PDFS  = [c for c in CONSTITUTIONS if c["source_type"] == "text_pdf"]

# Verify all preconditions before running the main pipeline steps.
checks = [
    (len(CONSTITUTIONS) == 38,             f"PDF files: {len(CONSTITUTIONS)}/38"),
    (bool(os.environ.get("TYPHOON_OCR_API_KEY")), "TYPHOON_OCR_API_KEY is set"),
    (INTERNET_OK,                           "Internet / API endpoint reachable"),
]

print("Setup Verification:")
all_ok = True
for ok, msg in checks:
    status = "OK" if ok else "FAIL"
    print(f"  [{status}]  {msg}")
    if not ok:
        all_ok = False

print(f"\n  Image PDFs (OCR required) : {len(IMAGE_PDFS)} documents (files 1-32)")
print(f"  Text PDFs  (direct extract): {len(TEXT_PDFS)} documents (files 33-38)")

cached_pages = list(OCR_CACHE_DIR.rglob("page_*.md"))
done_json    = list(PROCESSED_DIR.glob("const_*.json"))
if cached_pages:
    print(f"\n  OCR cache: {len(cached_pages)} pages already processed (resume enabled)")
if done_json:
    print(f"  Completed: {len(done_json)} JSON files already exist")

print("\nReady to run pipeline." if all_ok else "\nFix the issues above before proceeding.")

Setup Verification:
  [OK]  PDF files: 38/38
  [OK]  TYPHOON_OCR_API_KEY is set
  [OK]  Internet / API endpoint reachable

  Image PDFs (OCR required) : 32 documents (files 1-32)
  Text PDFs  (direct extract): 6 documents (files 33-38)

Ready to run pipeline.


---
## ส่วนที่ 2: กระบวนการ OCR สำหรับเอกสารภาพสแกน (ฉบับที่ 1 - 32)

ไฟล์ข้อมูลลำดับที่ 1 - 32 เป็นไฟล์ PDF ประเภทภาพสแกน ซึ่งไม่สามารถคัดลอกข้อความออกมาใช้งานได้โดยตรง ระบบจึงจำเป็นต้องประมวลผลรูปภาพเพื่อแปลงกลับมาเป็นข้อความภาษาไทย โดยเลือกใช้งานบริการ **Typhoon OCR API** (`typhoon-ocr==0.4.1`) ซึ่งเป็นเครื่องมือที่ฝึกอบรมมาเพื่อโครงสร้างเอกสารภาษาไทยโดยเฉพาะ

### ขั้นตอนการดำเนินงาน

1. สำหรับแต่ละเอกสาร ระบบจะนับจำนวนหน้าทั้งหมดก่อนโดยใช้โมดูล PyMuPDF
2. ดำเนินการต่อหน้า (Page-by-page) โดยส่งไปประมวลผลที่ Typhoon OCR API
3. ทำการบันทึกข้อความชั่วคราวลงในโฟลเดอร์ `ocr_cache/<document_id>/page_NNN.md` ทั้งนี้หากพบว่าไฟล์ฉบับนี้เคยถูกอ่านแล้ว ระบบจะข้ามการประมวลผลซ้ำเพื่อลดการร้องขอข้อมูลจากอินเทอร์เน็ต และทำให้เราสามารถรันการประมวลผลต่อจากเดิมได้ทันทีหาก Kaggle มีการปิดตัวเอง
4. เมื่อกระบวนการครบทุกหน้า ระบบจะรวมข้อความทั้งหมดเข้าด้วยกันภายใต้ฟิลด์ `full_text` และบันทึกผลเป็นไฟล์ JSON ในโฟลเดอร์ `processed/`

### ขีดจำกัดความเร็วการประมวลผล (Rate Limiting)

ข้อจำกัดในการร้องขอข้อมูลผ่าน API ของ Typhoon กำหนดไว้ที่ 20 ครั้งต่อนาที ระบบจึงมีการเว้นช่วงเวลาการหน่วงประมาณ 3 วินาที ระหว่างการส่งข้อมูลของแต่ละหน้า เพื่อรักษาสมดุลและป้องกันข้อผิดพลาดทางเทคนิคต่างๆ

### เวลาโดยประมาณ

สมมติให้ใช้เวลาประมวลผลตกหน้าละ 3 วินาที สำหรับเอกสารทั้ง 32 ฉบับซึ่งรวมประมาณ 900 - 1,200 หน้า คาดการณ์ว่าการดำเนินการในส่วนนี้อาศัยเวลาทั้งสิ้นประมาณ 8 - 14 ชั่วโมง

In [6]:
import json, re, time, logging
from datetime import datetime
from tqdm.notebook import tqdm
from tenacity import retry, stop_after_attempt, wait_exponential

# Configure logging so that progress is written both to the console
# and to a persistent log file for post-run review.
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.StreamHandler(),
        logging.FileHandler(str(LOG_DIR / "ocr_pipeline.log"), encoding="utf-8"),
    ]
)
log = logging.getLogger("ocr")

# Minimum seconds between API calls to respect the rate limit.
MIN_INTERVAL = 3.0


@retry(stop=stop_after_attempt(5), wait=wait_exponential(multiplier=2, min=4, max=60))
def _ocr_one_page(pdf_path: Path, page_num: int) -> str:
    '''Call the Typhoon OCR API for a single page and return the markdown text.'''
    from typhoon_ocr import ocr_document
    return ocr_document(pdf_or_image_path=str(pdf_path), page_num=page_num)


def _get_page_count(pdf_path: Path) -> int:
    '''Return the number of pages in the given PDF using PyMuPDF.'''
    try:
        import fitz
        doc = fitz.open(str(pdf_path))
        n = len(doc)
        doc.close()
        return n
    except Exception as e:
        log.error(f"Could not count pages in {pdf_path.name}: {e}")
        return 0


def _combine_pages(pages_data: list) -> str:
    '''
    Merge per-page OCR markdown into a single full-text string.
    We strip internal page-number tags and figure placeholders,
    then join pages with a horizontal rule separator.
    '''
    parts = []
    for p in pages_data:
        t = re.sub(r"<page_number>.*?</page_number>", "", p["raw_markdown"], flags=re.DOTALL)
        t = re.sub(r"<figure>.*?</figure>", "[image]", t, flags=re.DOTALL)
        t = re.sub(r"\n{3,}", "\n\n", t).strip()
        if t:
            parts.append(t)
    return "\n\n---\n\n".join(parts)


def ocr_constitution(meta: dict, skip_existing: bool = True, force: bool = False):
    '''
    Run the OCR pipeline for one constitutional document.

    Parameters
    ----------
    meta : dict
        Metadata dict from the CONSTITUTIONS list.
    skip_existing : bool
        If True and the output JSON already exists, return immediately.
    force : bool
        If True, ignore both the output JSON and page-level cache files
        and re-process everything from scratch.
    '''
    cid       = meta["id"]
    pdf_path  = PDF_DIR / meta["filename"]
    cache_dir = OCR_CACHE_DIR / cid
    out_path  = PROCESSED_DIR / f"{cid}.json"

    if not pdf_path.exists():
        log.warning(f"[{cid}] PDF not found: {pdf_path.name}")
        return None

    if out_path.exists() and skip_existing and not force:
        log.info(f"[{cid}] Already processed — skipping")
        return json.loads(out_path.read_text(encoding="utf-8"))

    cache_dir.mkdir(parents=True, exist_ok=True)
    total_pages = _get_page_count(pdf_path)
    if total_pages == 0:
        return None

    log.info(f"[{cid}] Starting OCR — {meta['name_short']} ({total_pages} pages)")

    pages_data = []
    last_req   = 0.0

    for pg in tqdm(range(1, total_pages + 1), desc=cid, unit="page", leave=False):
        cache_file = cache_dir / f"page_{pg:03d}.md"

        if cache_file.exists() and not force:
            # Page already cached; read from disk to avoid an unnecessary API call.
            md_text = cache_file.read_text(encoding="utf-8")
        else:
            # Enforce the rate limit before making the call.
            elapsed = time.time() - last_req
            if elapsed < MIN_INTERVAL:
                time.sleep(MIN_INTERVAL - elapsed)
            try:
                md_text = _ocr_one_page(pdf_path, pg)
                cache_file.write_text(md_text, encoding="utf-8")
                last_req = time.time()
            except Exception as e:
                log.error(f"  [page {pg}] Failed after retries: {e}")
                md_text = ""

        pages_data.append({
            "page_num":    pg,
            "raw_markdown": md_text,
            "char_count":  len(md_text),
            "has_figure":  "<figure>" in md_text.lower(),
        })

    full_text = _combine_pages(pages_data)

    result = {
        "id":                cid,
        "year_th":           meta["year_th"],
        "year_ce":           meta["year_ce"],
        "name_short":        meta["name_short"],
        "source_type":       "image_pdf",
        "processing_method": "typhoon-ocr-0.4.1",
        "processed_at":      datetime.now().isoformat(),
        "total_pages":       total_pages,
        "era":               meta["era"],
        "regime_type":       meta["regime_type"],
        "pages":             pages_data,
        "full_text":         full_text,
        "metadata": {
            "total_chars":        len(full_text),
            "total_words_approx": len(full_text.split()),
            "pages_with_figures": sum(1 for p in pages_data if p["has_figure"]),
        },
    }

    out_path.write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
    log.info(f"[{cid}] Done — {total_pages} pages, {len(full_text):,} characters")
    return result

print("OCR functions defined and ready.")

OCR functions defined and ready.


In [7]:
# Run the OCR pipeline for all 32 image-based PDFs.
# Documents that have already been processed (JSON file exists in processed/)
# are skipped automatically. Re-run this cell at any time to resume.

if not os.environ.get("TYPHOON_OCR_API_KEY"):
    raise RuntimeError("No API key — run Section 1.4 first")
if not IMAGE_PDFS:
    raise RuntimeError("No image PDFs found — check dataset configuration")

t0 = time.time()
print(f"Starting OCR pipeline: {datetime.now().strftime('%H:%M:%S')}")
print(f"  Documents: {len(IMAGE_PDFS)} | Cache directory: {OCR_CACHE_DIR}")
print()

ocr_results, ocr_failed = [], []

for i, meta in enumerate(IMAGE_PDFS, 1):
    print(f"[{i:02d}/{len(IMAGE_PDFS)}] {meta['id']} — {meta['name_short']}")
    try:
        r = ocr_constitution(meta, skip_existing=True)
        if r:
            ocr_results.append(r)
        else:
            ocr_failed.append(meta['id'])
    except Exception as e:
        print(f"   Error: {e}")
        ocr_failed.append(meta['id'])

elapsed = (time.time() - t0) / 3600
print(f"\n{'=' * 60}")
print(f"OCR pipeline complete: {len(ocr_results)}/{len(IMAGE_PDFS)} documents | {elapsed:.1f} hours")
if ocr_failed:
    print(f"  Failed: {ocr_failed}")
    print("  Re-run this cell to retry failed documents (page cache is preserved)")

Starting OCR pipeline: 09:48:07
  Documents: 32 | Cache directory: /kaggle/working/ocr_cache

[01/32] const_2475 — Constitution 2475


2026-04-08 09:48:07,316 [INFO] [const_2475] Starting OCR — Constitution 2475 (23 pages)


const_2475:   0%|          | 0/23 [00:00<?, ?page/s]

2026-04-08 09:48:14,581 [INFO] HTTP Request: POST https://api.opentyphoon.ai/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-08 09:48:24,798 [INFO] HTTP Request: POST https://api.opentyphoon.ai/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-08 09:48:34,926 [INFO] HTTP Request: POST https://api.opentyphoon.ai/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-08 09:48:43,451 [INFO] HTTP Request: POST https://api.opentyphoon.ai/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-08 09:48:51,378 [INFO] HTTP Request: POST https://api.opentyphoon.ai/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-08 09:48:57,866 [INFO] HTTP Request: POST https://api.opentyphoon.ai/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-08 09:49:06,488 [INFO] HTTP Request: POST https://api.opentyphoon.ai/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-08 09:49:14,135 [INFO] HTTP Request: POST https://api.opentyphoon.ai/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-08 09:49:22,113 [INFO] HTTP Request: POST https://api.opentyphoon.ai/v1/

[02/32] const_2482 — Constitution 2482


...

2026-04-08 11:48:29,715 [INFO] HTTP Request: POST https://api.opentyphoon.ai/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-08 11:48:38,198 [INFO] HTTP Request: POST https://api.opentyphoon.ai/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-08 11:48:44,370 [INFO] HTTP Request: POST https://api.opentyphoon.ai/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-08 11:48:44,374 [INFO] [const_2548] Done — 3 pages, 3,261 characters
2026-04-08 11:48:44,390 [INFO] [const_2549] Starting OCR — Constitution 2549 (interim) (14 pages)


[32/32] const_2549 — Constitution 2549 (interim)


const_2549:   0%|          | 0/14 [00:00<?, ?page/s]

2026-04-08 11:48:49,772 [INFO] HTTP Request: POST https://api.opentyphoon.ai/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-08 11:48:59,019 [INFO] HTTP Request: POST https://api.opentyphoon.ai/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-08 11:49:11,274 [INFO] HTTP Request: POST https://api.opentyphoon.ai/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-08 11:49:21,810 [INFO] HTTP Request: POST https://api.opentyphoon.ai/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-08 11:49:32,131 [INFO] HTTP Request: POST https://api.opentyphoon.ai/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-08 11:49:43,479 [INFO] HTTP Request: POST https://api.opentyphoon.ai/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-08 11:49:54,417 [INFO] HTTP Request: POST https://api.opentyphoon.ai/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-08 11:50:05,643 [INFO] HTTP Request: POST https://api.opentyphoon.ai/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-08 11:50:18,447 [INFO] HTTP Request: POST https://api.opentyphoon.ai/v1/


OCR pipeline complete: 32/32 documents | 2.1 hours


---
## ส่วนที่ 3: กระบวนการสกัดข้อความสำหรับเอกสารประเภทขัอความ (ฉบับที่ 33–38)

ไฟล์ลำดับที่ 33 ถึง 38 ประกอบด้วยรูปแบบอักษร Unicode ที่สมบูรณ์ในตัวเอง ซึ่งเอื้อต่อการคัดลอกได้โดยไม่ต้องผ่านกระบวนการ OCR ซึ่งนอกจากจะสะดวกรวดเร็วแล้ว (ใช้เวลาเพียงหลักนาที) ความถูกต้องของข้อความก็มักจะแม่นยำกว่าการสแกนด้วย OCR เสมอ

กลยุทธ์การดำเนินงานประกอบด้วย:
1. เริ่มจากวิธีหลักคือ การสกัดผ่านไลบรารี **PyMuPDF** (`fitz`) เนื่องจากประมวลผลอักขระภาษาไทยได้เสถียรที่สุด
2. สำรองการทำงาน (Fallback): ในกรณีข้อความสูญหาย หรืออ่านได้น้อยกว่า 500 ตัวอักษร ระบบจะสลับไปอ่านข้อมูลผ่านแพ็กเกจ **pdfplumber** แทน 

เมื่อได้ขัอความออกมาทั้งหมด ระบบจะทำ Post-processing เพื่อแก้ไขปัญหาช่องว่างเกิน จัดระเบียบการจัดวาง และแปลงเข้าสู่มาตรฐานสากล (Unicode NFC Form)

### ข้อค้นพบสำคัญ: ปัญหาสระอำ (สระอำหายหรือแยกร่าง)

เอกสารในกลุ่มปี 2550 จนถึงปัจจุบัน อาจพบว่ามีการฝังฟอนต์รูปแบบเก่าที่ใช้เทคนิคการเข้ารหัส **สระอำ** (U+0E33) ด้วยการใช้ต้วอักษรที่ไม่มีความกว้าง (Zero-width space) ประกอบกับสระอา เมื่อตัวอ่านทำการแยกข้อมูลออกมา ส่งผลให้คำว่า `จำนวน` ปรากฏกลายเป็น `จ านวน` 

กระบวนการนี้ได้ผนวกวิธีการแก้ปัญหานี้เข้าในฟังก์ชัน `_fix_sara_am()` เพื่อจับลักษณะแพทเทิร์นที่ผิดธรรมชาตินี้และแปลงโครงสร้างใหม่ให้กลับมาเป็นสระอำปกติสมบูรณ์

In [10]:
import unicodedata


def _fix_pua(text: str) -> str:
    """
    Convert legacy Thai Private Use Area (PUA) characters back to standard
    Unicode. Affected fonts (e.g. AngsanaUPC, CordiaUPC) stored above-vowels
    and tone marks in the F700-F71A range; PDF extraction exposes them raw.
    """
    if not text:
        return ""
    pua_map = {
        0xF700: 0x0E31, 0xF701: 0x0E34, 0xF702: 0x0E35, 0xF703: 0x0E36, 0xF704: 0x0E37,
        0xF705: 0x0E48, 0xF706: 0x0E49, 0xF707: 0x0E4A, 0xF708: 0x0E4B, 0xF709: 0x0E4C,
        0xF70A: 0x0E48, 0xF70B: 0x0E49, 0xF70C: 0x0E4A, 0xF70D: 0x0E4B, 0xF70E: 0x0E4C,
        0xF70F: 0x0E47, 0xF710: 0x0E31, 0xF711: 0x0E47, 0xF712: 0x0E47, 0xF713: 0x0E48,
        0xF714: 0x0E49, 0xF715: 0x0E4A, 0xF716: 0x0E4B, 0xF717: 0x0E4C, 0xF718: 0x0E38,
        0xF719: 0x0E39, 0xF71A: 0x0E3A,
    }
    return text.translate(pua_map)


def _normalize(text: str) -> str:
    """NFC normalization, strip control chars, collapse whitespace."""
    if not text:
        return ""
    text = unicodedata.normalize("NFC", text)
    text = re.sub(r"[\x00-\x08\x0b\x0c\x0e-\x1f]", "", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def _fix_sara_am(text: str) -> str:
    """
    Restore sara am (U+0E33) that was stored as [consonant] + SPACE + sara aa (U+0E32)
    in legacy Thai PDF font encodings. Example: 'จ านวน' -> 'จำนวน'.
    """
    # Pattern uses actual Unicode chars to avoid raw-string escape issues.
    SARA_AM_PATTERN = re.compile(
        '([ก-ฮ][็-๋]?) (า)',
    )
    return SARA_AM_PATTERN.sub(lambda m: m.group(1) + 'ำ', text)


def _remove_headers(text: str) -> str:
    """Remove Royal Gazette header/footer lines from every page."""
    patterns = [
        r"\u0E2B\u0E19\u0E49\u0E32\s*\d+\s*\u0E40\u0E25\u0E48\u0E21\s*\d+.*?\u0E23\u0E32\u0E0A\u0E01\u0E34\u0E08\u0E08\u0E32\u0E19\u0E38\u0E40\u0E1A\u0E01\u0E29\u0E32[^\n]*",
        r"\u0E23\u0E32\u0E0A\u0E01\u0E34\u0E08\u0E08\u0E32\u0E19\u0E38\u0E40\u0E1A\u0E01\u0E29\u0E32.*?\u0E40\u0E25\u0E48\u0E21\s*\d+.*?\u0E2B\u0E19\u0E49\u0E32\s*\d+[^\n]*",
        r"\u0E40\u0E25\u0E48\u0E21\s*\d+\s*\u0E15\u0E2D\u0E19\u0E17\u0E35\u0E48\s*\d+[^\n]*\u0E23\u0E32\u0E0A\u0E01\u0E34\u0E08\u0E08\u0E32\u0E19\u0E38\u0E40\u0E1A\u0E01\u0E29\u0E32[^\n]*",
        r"^\s*-\s*\d+\s*-\s*$",
        r"^\s*\d+\s*$",
    ]
    for pattern in patterns:
        text = re.sub(pattern, "", text, flags=re.MULTILINE | re.IGNORECASE)
    return re.sub(r"\n{3,}", "\n\n", text).strip()


def _extract_text_smart(pdf_path: Path):
    """
    Try PyMuPDF first; fall back to pdfplumber if text is too short.
    Returns (list of page dicts, method_name).
    Each page dict has keys: 'page_num' (int) and 'text' (str).
    """
    try:
        import fitz
        doc = fitz.open(str(pdf_path))
        pages = [
            {"page_num": i + 1, "text": page.get_text("text")}
            for i, page in enumerate(doc)
        ]
        doc.close()
        if sum(len(p.get("text", "")) for p in pages) > 500:
            return pages, "pymupdf"
    except Exception:
        pass

    import pdfplumber
    with pdfplumber.open(str(pdf_path)) as pdf:
        pages = [
            {"page_num": i + 1, "text": (p.extract_text() or "")}
            for i, p in enumerate(pdf.pages)
        ]
    return pages, "pdfplumber"


def extract_constitution(meta: dict, skip_existing: bool = True):
    """
    Extract and clean text from one text-based PDF, save to processed/.

    Post-processing pipeline per page:
    1. _fix_pua      — map PUA characters to standard Unicode
    2. _normalize    — NFC, strip control chars, collapse whitespace
    3. _fix_sara_am  — restore split sara am vowel
    4. _remove_headers — strip Royal Gazette headers and page numbers
    """
    cid      = meta["id"]
    pdf_path = PDF_DIR / meta["filename"]
    out_path = PROCESSED_DIR / f"{cid}.json"

    if not pdf_path.exists():
        print(f"  [{cid}] PDF not found: {pdf_path.name}")
        return None

    if out_path.exists() and skip_existing:
        print(f"  [{cid}] Already processed — skipping")
        return json.loads(out_path.read_text(encoding="utf-8"))

    pages_data, method = _extract_text_smart(pdf_path)

    # Apply the full post-processing pipeline to each page.
    for p in pages_data:
        text = _fix_pua(p.get("text", ""))
        text = _normalize(text)
        text = _fix_sara_am(text)
        text = _remove_headers(text)
        p["text"] = text  # overwrite in place with the same key

    full_text = _normalize(
        "\n\n".join(p["text"] for p in pages_data if p.get("text"))
    )

    result = {
        "id":                cid,
        "year_th":           meta["year_th"],
        "year_ce":           meta["year_ce"],
        "name_short":        meta["name_short"],
        "source_type":       "text_pdf",
        "processing_method": method,
        "processed_at":      datetime.now().isoformat(),
        "total_pages":       len(pages_data),
        "era":               meta["era"],
        "regime_type":       meta["regime_type"],
        "pages":             pages_data,
        "full_text":         full_text,
        "metadata": {
            "total_chars":        len(full_text),
            "total_words_approx": len(full_text.split()),
            "empty_pages":        sum(1 for p in pages_data if not p.get("text", "").strip()),
        },
    }

    out_path.write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"  [{cid}] Done — {len(pages_data)} pages, {len(full_text):,} chars [{method}]")
    return result


print("Text extraction functions defined and ready.")

Text extraction functions defined and ready.


In [11]:
# Run text extraction for all 6 text-based PDFs.

print(f"Starting text extraction: {len(TEXT_PDFS)} documents")
print()

txt_results = []
for meta in TEXT_PDFS:
    r = extract_constitution(meta, skip_existing=True)
    if r:
        txt_results.append(r)

print(f"\nText extraction complete: {len(txt_results)}/{len(TEXT_PDFS)} documents")

Starting text extraction: 6 documents

  [const_2550] Done — 127 pages, 233,697 chars [pymupdf]
  [const_2554a] Done — 6 pages, 9,606 chars [pymupdf]
  [const_2554b] Done — 3 pages, 3,675 chars [pymupdf]
  [const_2557] Done — 17 pages, 38,336 chars [pymupdf]
  [const_2560] Done — 90 pages, 206,611 chars [pymupdf]
  [const_2564] Done — 4 pages, 5,532 chars [pymupdf]

Text extraction complete: 6/6 documents


---
## ส่วนที่ 4: สรุปผลลัพธ์กระบวนการ (Results Summary)

ส่วนนี้มีวัตถุประสงค์เพื่อเรียกอ่านไฟล์ JSON ฉบับสมบูรณ์จากไดเรกทอรี `processed/` และนำมาแสดงผลเป็นตารางสถิติสรุปภาพรวม เพื่อตรวจสอบขัอมูลขั้นสุดท้ายว่าการทำงานทั้งหมดสัมฤทธิ์ผลหรือไม่ เอกสารมีความครบถ้วน 38 ฉบับหรือไม่ และตรวจสอบขนาดของจำนวนคำโดยละเอียดต่อไป

In [12]:
import pandas as pd
from IPython.display import display

all_json = sorted(PROCESSED_DIR.glob("const_*.json"))
rows = []

for jf in all_json:
    try:
        d = json.loads(jf.read_text(encoding="utf-8"))
        rows.append({
            "Year (B.E.)":    d.get("year_th", ""),
            "Document":       d.get("name_short", jf.stem),
            "Type":           "Image PDF" if d.get("source_type") == "image_pdf" else "Text PDF",
            "Pages":          d.get("total_pages", 0),
            "Words (approx)": d.get("metadata", {}).get("total_words_approx", 0),
            "Method":         d.get("processing_method", ""),
        })
    except Exception as e:
        print(f"  Warning: could not read {jf.name}: {e}")

if rows:
    df = pd.DataFrame(rows).sort_values("Year (B.E.)").reset_index(drop=True)

    print(f"Documents processed: {len(df)}/38")
    print(f"  Image PDF (OCR)    : {len(df[df['Type'] == 'Image PDF'])} documents")
    print(f"  Text PDF (extract) : {len(df[df['Type'] == 'Text PDF'])} documents")
    print(f"  Total words        : {df['Words (approx)'].sum():,} (approximate)")
    print()

    display(
        df.drop(columns=["Method"])
          .style
          .format({"Words (approx)": "{:,}"})
          .background_gradient(subset=["Words (approx)"], cmap="Blues")
          .hide(axis="index")
    )
else:
    print("No processed files found. Run Sections 2 and 3 first.")

2026-04-08 12:49:41,221 [INFO] NumExpr defaulting to 4 threads.


Documents processed: 38/38
  Image PDF (OCR)    : 32 documents
  Text PDF (extract) : 6 documents
  Total words        : 61,301 (approximate)



Year (B.E.),Document,Type,Pages,Words (approx)
2475,Constitution 2475,Image PDF,23,987
2482,Constitution 2482,Image PDF,2,78
2483,Constitution 2483,Image PDF,3,108
2485,Constitution 2485,Image PDF,3,109
2489,Constitution 2489,Image PDF,51,"2,021"
2490,Constitution 2490 (interim),Image PDF,32,"1,313"
2490,Constitution 2490 (amended),Image PDF,4,151
2491,Constitution 2491 (No.2),Image PDF,8,387
2491,Constitution 2491 (No.3),Image PDF,2,67
2492,Constitution 2492,Image PDF,80,"3,238"


---
## ส่วนที่ 5: บันทึกข้อมูลข้อบกพร่องและแก้ไข (Data Quality Notes)

จากการตรวจสอบเชิงคุณภาพอย่างละเอียด พบว่าข้อมูลบางส่วนยังมีการจัดเรียงที่ไม่สอดคล้องกัน โดยคณะผู้จัดทำได้ทำการตรวจทานและทำการแก้ไขไว้แล้ว

### ประเด็นที่ 1 — รูปแบบสระอำที่ไม่สมบูรณ์ในเอกสารรัฐธรรมนูญข้อความ

**ระดับความรุนแรง:** สูง (มีผลต่อความหมายของคำ)  
**ไฟล์ที่ได้รับผลกระทบ:** const_2550, const_2554a, const_2554b, const_2557, const_2560, const_2564  
**สาเหตุ:** เอกสารต้นฉบับได้มีการตั้งค่าการใช้ชุดฟอนต์ที่ใช้การเข้ารหัสอักษรเก่าที่มีปัญหา  
**การแก้ไข:** กระบวนการ Post-processing โดยฟังก์ชัน `_fix_sara_am()` ตรวจหาช่องว่างระหว่างพยัญชนะกับสระอา และแทนที่กลับให้ถูกต้อง เช่น แทนที่ "จ านวน" ด้วย "จำนวน" 

### ประเด็นที่ 2 — ข้อความอ้างอิงราชกิจจานุเบกษาและเลขหน้าส่วนเกิน

**ระดับความรุนแรง:** ปานกลาง  
**ไฟล์ที่ได้รับผลกระทบ:** ไฟล์ประเภทข้อความทุกไฟล์  
**สาเหตุ:** แต่ละหน้าของเอกสารต้นฉบับมีข้อความ "ราชกิจจานุเบกษา" พร้อมเลขหน้าและชุดลำดับแทรกอยู่  
**หมายเหตุ:** สำหรับปี 2564 ระบบของผู้อ่านดึงข้อมูลผิดเพี้ยนไปทำให้เกิดข้อความ "้หนา ๑ ่เลม ๑๓๘" สลับด้าน  
**การแก้ไข:** จำเป็นต้องมีการใช้สคริปต์ทำความสะอาดเฉพาะส่วน (RegEx-based cleaner) เพื่อทำการตัดคำส่วนขยายและข้อความส่วนหัวหรือส่วนท้ายเหล่านี้ออกไป ก่อนการใช้งานในขั้นวิเคราะห์ข้อมูลต่อไป